# SpliSync Inference Workflow Demo

This notebook demonstrates the inference pipeline for **SpliSync**, a method for correcting splice sites in long-read RNA alignments.

In [ ]:
%%bash
# Create and activate the conda environment
conda env create -f splisync_env.yml
conda activate splisync_env.yml

# init git lfs (large file storage)
git lfs install

# Compile addXS binary
g++ -c -o addXS.o -I./samtools-0.1.19 -L./samtools-0.1.19 -Wall -O3  AddXS.cpp -lbam -lz -lm -lpthread
g++ -o addXS -I./samtools-0.1.19 -L./samtools-0.1.19 -Wall -O3  addXS.o -lbam -lz -lm -lpthread

## 1. Configuration & Input Variables

In [ ]:
import os

# Set these paths according to your local environment
os.environ['SPLISYNC_DIR'] = "." # path/to/SpliSync
os.environ['REF_FASTA'] = "/path/to/ref_fasta.fa"
os.environ['MODEL_CKPT'] = "${SPLISYNC_DIR}/model_weights/cdna_r9.ckpt"
os.environ['INPUT_BAM'] = "${SPLISYNC_DIR}/test_data/reads_aligned_reads.50k.bam"

# Intermediate and output file prefixes
prefix = os.environ['INPUT_BAM'].replace('.bam', '')
os.environ['PREFIX'] = prefix

os.environ['MERGED_BED'] = f"{prefix}.bam.merge"
os.environ['DATA_DIR'] = f"inferencedata_{prefix}.bam.merge"
os.environ['PREDICTIONS'] = f"{os.environ['DATA_DIR']}.predictions.tsv"
os.environ['BED12_OUT'] = f"{prefix}.bed12"
os.environ['INTRON_MARKS'] = f"{prefix}.intron_marks"
os.environ['UPDATED_INTRONS'] = f"{prefix}.updated_{os.environ['PREDICTIONS']}_threshold21"
os.environ['CORRECTED_BED'] = f"{prefix}.corrected.bed"
os.environ['CLEAN_BED'] = f"{prefix}.corrected.cleaned.bed"
os.environ['CLEAN_BAM'] = f"{prefix}.corrected.cleaned.bam"

print(f"Workflow initialized for: {os.environ['INPUT_BAM']}")
print(f"Output prefix set to: {os.environ['PREFIX']}")

## 2. Preprocessing & Data Curation
In this step, we index the reference genome, merge the input BAM file (ensure your input BAM is already sorted and indexed), and extract features for the deep learning model.

In [ ]:
%%bash
echo "[1/5] Preparing reference and BAM files..."
samtools faidx "$REF_FASTA"
samtools index "$INPUT_BAM"
bedtools merge -i "$INPUT_BAM" > "$MERGED_BED"

echo "[2/5] Curating dataset for inference..."
python "${SPLISYNC_DIR}/curate_dataset.inference.py" "$INPUT_BAM" "$REF_FASTA"

## 3. SpliSync Model Inference
Run the SpliSync model on the curated dataset using the specified checkpoint to generate splice site predictions.
Output format:
chrom  coord  score  is_intron_end  strand

In [ ]:
%%bash
echo "[3/5] Running SpliSync inference..."
python "${SPLISYNC_DIR}/run_splisync.py" \
    --data_dir "$DATA_DIR" \
    --model_path "$MODEL_CKPT"

## 4. BED Block Correction
Convert the BAM file to BED12 format, extract intron marks, apply the model's predictions to shift splice boundaries, and clean up any resulting zero-length exon blocks.

In [ ]:
%%bash
echo "[4/5] Applying predictions and correcting BED blocks..."

# Convert BAM to BED12
bedtools bamtobed -i "$INPUT_BAM" -bed12 > "$BED12_OUT"

# Extract introns and apply model predictions
# Output format: 
# index  chrom  coord  is_intron_end  strand
python "${SPLISYNC_DIR}/bed12_to_introns.py" "$BED12_OUT" > "$INTRON_MARKS"

# Update intron start / end coordinates using positions predicted by SpliSync
python "${SPLISYNC_DIR}/update_introns.py" "$INTRON_MARKS" "$PREDICTIONS" -o "$UPDATED_INTRONS"

# Correct the BED12 blocks based on updated splice sites
python "${SPLISYNC_DIR}/correct_bed_blocks.py" "$BED12_OUT" "$UPDATED_INTRONS" -o "$CORRECTED_BED"

# Clean up zero-length blocks
# Output: cleaned.bed, filtered_indices
python "${SPLISYNC_DIR}/remove_zero_blocks.py" "$CORRECTED_BED"

echo "BED Correction Complete."

## 5. BAM Reconstruction & Tag Transfer
Convert the corrected BED12 file back to BAM format. We also extract the updated FASTA sequences, add `XS` tags for downstream transcriptome analysis, and transfer the original BAM alignment tags to our finalized output.

In [ ]:
%%bash
echo "[5/5] Reconstructing corrected BAM file..."

# Extract FASTA sequences in the background
bedtools getfasta -fi "$REF_FASTA" -bed "$CLEAN_BED" -split -bedOut > "${CLEAN_BED}.getfasta" &

# Convert corrected BED12 back to BAM
bedToBam -i "$CLEAN_BED" -g "${REF_FASTA}.fai" -bed12 > "$CLEAN_BAM"
samtools index "$CLEAN_BAM"

# Add XS tags for splicing analysis
samtools view -h "$CLEAN_BAM" | "${SPLISYNC_DIR}/addXS" "$REF_FASTA" > "${CLEAN_BAM}.addxs"

# Wait for background processes (getfasta) to finish
wait

# Transfer original BAM tags to the newly corrected BAM
python "${SPLISYNC_DIR}/transfer_bam_tags.py" \
    "$INPUT_BAM" \
    "$CLEAN_BAM" \
    "${CORRECTED_BED}.filtered_indices" \
    "${CLEAN_BED}.getfasta" \
    "${CLEAN_BAM}.addxs"

echo "Workflow complete! Corrected BAM: ${CLEAN_BAM%.*}.withtags.bam"